<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/RAW_TrainingModel_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1 — Building a Simple Tokenizer
A tokenizer takes a raw string of text and breaks it into smaller pieces (tokens). For example, Byte Pair Encoding (BPE) which breaks words into subwords — like "unhappiness" → "un", "happi", "ness".

In [ ]:
# Step 1: A simple word-level tokenizer

text = "I love code and I love AI"

# Split into words (simplest possible tokenization)
tokens = text.lower().split()
print("Tokens:", tokens)

Tokens: ['i', 'love', 'code', 'and', 'i', 'love', 'ai']


# Step 2 — Building a Vocabulary
A neural network can't read the word "code". It only understands numbers. So we need a lookup table that assigns every unique word an integer ID.

In [ ]:
# Step 2: Build a vocabulary (unique words → integer IDs)

# Get unique words, sorted for consistency
vocab = sorted(set(tokens))
print("Vocabulary:", vocab)

# Create two dictionaries — one for each direction
word_to_id = {word: idx for idx, word in enumerate(vocab)}
id_to_word = {idx: word for word, idx in word_to_id.items()}

print("\nWord → ID mapping:")
for word, idx in word_to_id.items():
    print(f"  '{word}' → {idx}")

Vocabulary: ['ai', 'and', 'code', 'i', 'love']

Word → ID mapping:
  'ai' → 0
  'and' → 1
  'code' → 2
  'i' → 3
  'love' → 4


In [ ]:
# Convert tokens to IDs
token_ids = [word_to_id[token] for token in tokens]
print("Token IDs:", token_ids)

# Convert to a PyTorch tensor
import torch
token_ids_tensor = torch.tensor(token_ids)
print("As tensor:", token_ids_tensor)

Token IDs: [3, 4, 2, 1, 3, 4, 0]
As tensor: tensor([3, 4, 2, 1, 3, 4, 0])


# Step 3 — The Embedding Layer
Here's where numbers transform into meaning. Right now we have [3, 4, 2, 1, 3, 4, 0] — just flat integers. These don't capture any relationship between words. The number 3 isn't "closer" to 4 than to 0 in any meaningful way.

The embedding layer is essentially a big table with one row per word in our vocabulary, where each row is a vector of numbers

### What nn.Embedding(vocab_size, d_model) does:
It creates a matrix of random numbers with shape (vocab_size, d_model). For GPT with 50,000 tokens and 768 dimensions, it would create a (50,000, 768) table. That's it. Nothing fancy — just a big table of random numbers, one row per word.

```code
Think of it as a massive spreadsheet:
Row 0    (token "the"):  [0.23, 0.87, 0.11, ... 768 numbers]
Row 1    (token "of"):   [0.56, 0.32, 0.78, ... 768 numbers]
Row 2    (token "and"):  [0.41, 0.65, 0.29, ... 768 numbers]
...
Row 49999 (token "zzz"): [0.12, 0.44, 0.93, ... 768 numbers]
```

*Because nn.Embedding is part of nn.Module, PyTorch tracks gradients on that table. So during training, when backpropagation happens, it doesn't update all 50,000 rows. It only updates the rows that were actually looked up in that training step. If your sentence only used words 3, 4, 2, 1, and 0 — only those 5 rows get nudged. The other 49,995 rows stay untouched for that step.*


In [ ]:
# Step 3: Embedding layer

vocab_size = len(vocab)     # 5 unique words
d_model = 4                 # each word gets a 4-dimensional vector

# nn.Embedding is just a fancy lookup table
embedding_layer = torch.nn.Embedding(vocab_size, d_model)

print("Embedding table (randomly initialized):")
print(embedding_layer.weight)
print("Shape:", embedding_layer.weight.shape)

Embedding table (randomly initialized):
Parameter containing:
tensor([[ 1.1783, -0.7788, -0.4714,  1.0542],
        [-1.0532,  0.0636,  0.2135,  0.7017],
        [ 0.7350,  0.2881, -0.7473,  0.1544],
        [-0.4273, -0.3534,  2.7149, -0.0090],
        [ 0.4683,  0.9351,  0.7040,  1.3154]], requires_grad=True)
Shape: torch.Size([5, 4])


### Now what happens when you call embedding_layer(token_ids_tensor)?

This is just a lookup operation. It does NOT create a new table. It goes to the existing table and picks out the rows you asked for.
Exactly what's happening:
```python
# Our token IDs:    [3, 4, 2, 1, 3, 4, 0]
#                    i love code and  i love ai

# What embedding_layer(token_ids_tensor) actually does:
# "Give me row 3" → grabs the vector for 'i'
# "Give me row 4" → grabs the vector for 'love'
# "Give me row 2" → grabs the vector for 'code'
# "Give me row 1" → grabs the vector for 'and'
# "Give me row 3" → grabs the vector for 'i'    (SAME row as before!)
# "Give me row 4" → grabs the vector for 'love' (SAME row as before!)
# "Give me row 0" → grabs the vector for 'ai'
```

The shape should be (7, 4) — 7 tokens in our sentence, each now represented by a 4-dimensional vector. Here's the critical thing: word "i" at position 0 and word "i" at position 4 have the exact same embedding. Because they're the same word, they look up the same row in the table.

And that's a problem. Think about this sentence: "The bank by the river" vs "I went to the bank." The word "bank" should mean different things based on position and context. Embeddings alone can't handle this — they just say "bank is bank."

Context comes from attention. But there's still the position problem — the model needs to know that the first "i" and the second "i" are in different positions.
That's what positional embeddings solve.

In [ ]:
# Look up embeddings for our sentence
token_embeddings = embedding_layer(token_ids_tensor)

print("Input IDs:", token_ids_tensor)
print("\nToken embeddings:")
print(token_embeddings)
print("Shape:", token_embeddings.shape)

Input IDs: tensor([3, 4, 2, 1, 3, 4, 0])

Token embeddings:
tensor([[-0.4273, -0.3534,  2.7149, -0.0090],
        [ 0.4683,  0.9351,  0.7040,  1.3154],
        [ 0.7350,  0.2881, -0.7473,  0.1544],
        [-1.0532,  0.0636,  0.2135,  0.7017],
        [-0.4273, -0.3534,  2.7149, -0.0090],
        [ 0.4683,  0.9351,  0.7040,  1.3154],
        [ 1.1783, -0.7788, -0.4714,  1.0542]], grad_fn=<EmbeddingBackward0>)
Shape: torch.Size([7, 4])


# Step 4 — Positional Embeddings
We generate a unique "position signature" for each position using sine and cosine waves of different frequencies. Position 0 gets one pattern, position 1 gets a different pattern, position 6 gets yet another. These patterns are crafted so the model can easily figure out both absolute position ("I'm at position 3") and relative distance ("these two words are 2 positions apart").

In [ ]:
# Step 4: Positional Encoding using sin/cos

import math

def create_positional_encoding(max_len, d_model):
    """
    Creates the sin/cos positional encoding table.
    max_len: longest sentence we'll handle
    d_model: embedding dimension (must match token embeddings)
    """
    pe = torch.zeros(max_len, d_model)

    for pos in range(max_len):          # for each position in the sentence
        for i in range(0, d_model, 2):  # for each pair of dimensions
            # The "wavelength" changes with dimension
            denominator = 10000 ** (i / d_model)

            pe[pos, i]     = math.sin(pos / denominator)  # even dimensions
            pe[pos, i + 1] = math.cos(pos / denominator)  # odd dimensions

    return pe

# Create positional encodings for up to 10 positions
pe = create_positional_encoding(max_len=10, d_model=4)

print("Positional encoding for first 7 positions:")
print(pe[:7])
print("Shape:", pe[:7].shape)

Positional encoding for first 7 positions:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [-0.7568, -0.6536,  0.0400,  0.9992],
        [-0.9589,  0.2837,  0.0500,  0.9988],
        [-0.2794,  0.9602,  0.0600,  0.9982]])
Shape: torch.Size([7, 4])


### Why adding the positional embeddings to the token embeddings instead of stacking them like (token_embeddings, position,embeddings)?

The deeper reason is about efficiency and information blending. When you add, the position information gets baked into the same 4 numbers that carry meaning. The model is forced to learn embeddings where meaning and position can coexist in the same space. This keeps the model smaller (no extra dimensions) and importantly, when attention later compares two words, it's comparing one vector that contains both "what" and "where" — not two separate pieces it has to figure out how to combine.

In [ ]:
# Step 5: Combine token embeddings + positional encodings

# Take only the positions we need (7 tokens)
positional_encoding = pe[:len(tokens)]

# THE KEY OPERATION: add them together
final_embeddings = token_embeddings + positional_encoding

print("Token embeddings (what the word means):")
print(token_embeddings[0], "← first 'i'")
print(token_embeddings[4], "← second 'i'")

print("\nPositional encodings (where the word is):")
print(positional_encoding[0], "← position 0")
print(positional_encoding[4], "← position 4")

print("\nFinal embeddings (meaning + position):")
print(final_embeddings[0], "← 'i' at position 0")
print(final_embeddings[4], "← 'i' at position 4")

print("\nAre the two 'i' embeddings the same?",
      torch.equal(final_embeddings[0], final_embeddings[4]))

print("final_embeddings shape: ", final_embeddings.shape)

Token embeddings (what the word means):
tensor([-0.4273, -0.3534,  2.7149, -0.0090], grad_fn=<SelectBackward0>) ← first 'i'
tensor([-0.4273, -0.3534,  2.7149, -0.0090], grad_fn=<SelectBackward0>) ← second 'i'

Positional encodings (where the word is):
tensor([0., 1., 0., 1.]) ← position 0
tensor([-0.7568, -0.6536,  0.0400,  0.9992]) ← position 4

Final embeddings (meaning + position):
tensor([-0.4273,  0.6466,  2.7149,  0.9910], grad_fn=<SelectBackward0>) ← 'i' at position 0
tensor([-1.1841, -1.0071,  2.7549,  0.9902], grad_fn=<SelectBackward0>) ← 'i' at position 4

Are the two 'i' embeddings the same? False
final_embeddings shape:  torch.Size([7, 4])


In [ ]:
# import torch
# print(torch.__version__)

In [ ]:
# Tensor: 3 words, each represented by 4 numbers
# sentence = torch.tensor([
#     [1.0, 0.5, 0.3, 0.8], # word-1: "I"
#     [0.2, 0.9, 0.7, 0.1], # word-2: "love"
#     [0.6, 0.4, 0.2, 0.5]  # word-3: "code"
# ])

# print("Shape: ", sentence.shape)
# print(sentence)

# Step-5: Creating Query, Key and Value Matrices

Every word in a sentence has one embedding — its identity. But during
attention, each word needs to play three different roles:

- Query (Q): "What am I looking for?" — what this word wants from others
- Key (K):   "What do I offer?" — what this word advertises about itself  
- Value (V): "What do I actually contain?" — the real information this word shares

We create Q, K, V by multiplying the same embedding by three SEPARATE
weight matrices (W_Q, W_K, W_V). Same word, three different lenses.
These weight matrices start random and get adjusted during training.

In [ ]:
torch.manual_seed(42) # To get the same random numbers

# d_model = 4 # embedding size (each word is 4 numbers) - already defined above
d_k = 3       # project into 3 dimensions for Q, K, V

# Weight Matrices
W_Q = torch.randn(d_model, d_k) # shape: (4, 3)
W_K = torch.randn(d_model, d_k) # shape: (4, 3)
W_V = torch.randn(d_model, d_k) # shape: (4, 3)

print("W_Q Shape: ", W_Q.shape)
print("W_K Shape: ", W_K.shape)
print("W_V Shape: ", W_V.shape)

W_Q Shape:  torch.Size([4, 3])
W_K Shape:  torch.Size([4, 3])
W_V Shape:  torch.Size([4, 3])


## What does the Q, K, V vector actually contain?

Each number in a Q, K, or V vector represents some abstract learned
feature — NOT something human-readable like "this number means noun"
or "this number means happiness."

Think of it like coordinates. The embedding [5.6, -1.3, 2.4] is a
POINT in 3D space. The exact numbers don't matter individually. What
matters is WHERE that point sits relative to other points.

Q vector = a point representing "what I'm searching for"
K vector = a point representing "what I'm offering"
V vector = a point representing "what I'll actually share"

Same word, projected to three different locations in space. The model
discovers what these dimensions mean during training — we never define
them. With random weights, the vectors are meaningless. After training,
words that should interact end up with Q and K vectors pointing in
similar directions.

In [ ]:
# final_embeddings shape: (7, 4) - 7 words, 4-dim embeddings
# W_Q shape: (4, 3) - projects 4-dim into 3-dim

Q = final_embeddings @ W_Q # (3, 4) @ (4, 3) = (3, 3)
K = final_embeddings @ W_K # (3, 4) @ (4, 3) = (3, 3)
V = final_embeddings @ W_V # (3, 4) @ (4, 3) = (3, 3)

print("Q shape:", Q.shape)  # (7, 3) — 7 tokens, 3-dim queries
print("K shape:", K.shape)
print("V shape:", V.shape)

print("Q (Queries):\n", Q)
print("K (Keys):\n", K)
print("V (Values):\n", V)

Q shape: torch.Size([7, 3])
K shape: torch.Size([7, 3])
V shape: torch.Size([7, 3])
Q (Queries):
 tensor([[ 6.2650, -1.9830,  1.8347],
        [ 2.9764, -0.7049,  2.2357],
        [-0.7734,  1.4370,  1.0078],
        [ 0.4721,  1.6773,  1.4481],
        [ 5.7174, -0.2497,  1.9832],
        [ 2.3991, -0.6748,  1.8789],
        [-0.0153,  1.2724,  1.6481]], grad_fn=<MmBackward0>)
K (Keys):
 tensor([[-1.2540, -0.7858,  5.4107],
        [ 3.9099, -1.5574,  2.7166],
        [ 3.0522, -2.7802, -1.7970],
        [-0.9190, -0.4239,  1.8268],
        [-3.7096, -1.7231,  4.8612],
        [ 1.6336,  1.1160,  4.3405],
        [ 2.8955, -1.6431,  0.3343]], grad_fn=<MmBackward0>)
V (Values):
 tensor([[ 0.5587,  0.9165, -0.4023],
        [ 2.0114, -1.8312, -0.5015],
        [ 1.4968, -1.6038, -0.3305],
        [ 1.4318, -1.5553, -2.4438],
        [ 0.7388,  0.9865, -1.6622],
        [ 1.5643, -1.8891, -1.5713],
        [ 1.9865, -2.3430, -1.3135]], grad_fn=<MmBackward0>)


# Step-6: Computing Attention Scores

Formula:
```code
Attention (Q, K, V) = softmax(QK^T / √d_k) x V
```

### Part-1: QK^T (The Dot Product)

The dot product measures how much two vectors point in the SAME direction.

- Same direction     → large positive number → "highly relevant"
- Perpendicular      → near zero             → "not relevant"  
- Opposite direction → negative number       → "ignore this word"

Q @ K.T compares every word's Query against every word's Key. The
result is a matrix where score[i][j] answers: "how relevant is word j
to what word i is looking for?"

High score = strong match between what word i wants and what word j offers.
Low/negative score = mismatch, word i will mostly ignore word j.

If these scores are wrong (e.g. "I" ignores "AI" in "I love AI"), the
loss function catches the bad prediction and backpropagation adjusts
W_Q and W_K so their Q and K vectors align better next time.

In [ ]:
scores = Q @ K.T # (7, 3) @ (3, 7) = (7, 7)

print("Raw attention scores: ")
print(scores)

Raw attention scores: 
tensor([[  3.6292,  32.5682,  21.3385,  -1.5654, -10.9045,  15.9855,  22.0119],
        [  8.9184,  18.8089,   7.0269,   1.6477,   1.0419,  13.7799,  10.5238],
        [  5.2934,  -2.5241,  -8.1666,   1.9427,   5.2920,   4.7145,  -4.2635],
        [  5.9249,   3.1677,  -5.8242,   1.5004,   2.3979,   8.9285,  -0.9047],
        [  3.7572,  28.1309,  14.5812,  -1.5257, -11.1380,  17.6698,  17.6279],
        [  7.6881,  15.5353,   5.8223,   1.5137,   1.3971,  11.3216,   8.6834],
        [  7.9369,   2.4357,  -6.5461,   2.4856,   5.8764,   8.5488,  -1.5841]],
       grad_fn=<MmBackward0>)


### Part-2: Scaling by √d_k

The dot product Q @ K.T gives raw attention scores. But these scores
grow larger as d_k (dimension size) increases — simply because more
numbers are being added together, not because words are more relevant.

Large numbers break softmax — it pushes 99% attention to one word and
ignores everything else. Dividing by √d_k shrinks scores back to a
gentle range so softmax can spread attention across multiple words.

Scaling keeps softmax in its "useful zone" where it can spread attention across multiple words, instead of its "extreme zone" where it picks one word and ignores everything else.

In [ ]:
import math
d_k = 3 # dimension of our key vectors
scaled_scores = scores / math.sqrt(d_k)

print("Scaled scores: ")
print(scaled_scores)

Scaled scores: 
tensor([[ 2.0953, 18.8033, 12.3198, -0.9038, -6.2957,  9.2292, 12.7086],
        [ 5.1490, 10.8593,  4.0570,  0.9513,  0.6016,  7.9559,  6.0759],
        [ 3.0561, -1.4573, -4.7150,  1.1216,  3.0553,  2.7219, -2.4615],
        [ 3.4208,  1.8289, -3.3626,  0.8663,  1.3844,  5.1549, -0.5223],
        [ 2.1692, 16.2414,  8.4184, -0.8808, -6.4305, 10.2017, 10.1775],
        [ 4.4387,  8.9693,  3.3615,  0.8739,  0.8066,  6.5365,  5.0134],
        [ 4.5824,  1.4062, -3.7794,  1.4350,  3.3928,  4.9356, -0.9146]],
       grad_fn=<DivBackward0>)


### Part-3: Softmax

Raw attention scores are just numbers — some positive, some negative,
no fixed range. Softmax converts them into probabilities between 0 and 1
where each row sums to exactly 1.0.

Now each row is a pie chart: "I give 40% attention here, 35% there,
25% there." This tells us exactly how much each word cares about
every other word.

In [ ]:
attention_weights = torch.softmax(scaled_scores, dim=-1)

print("Attention wights:")
print(attention_weights)
print("\nEach row sums to: ", attention_weights.sum(dim=-1))

Attention wights:
tensor([[5.5228e-08, 9.9616e-01, 1.5227e-03, 2.7521e-09, 1.2531e-11, 6.9244e-05,
         2.2462e-03],
        [3.1018e-03, 9.3658e-01, 1.0407e-03, 4.6618e-05, 3.2860e-05, 5.1356e-02,
         7.8369e-03],
        [3.4783e-01, 3.8125e-03, 1.4669e-04, 5.0258e-02, 3.4755e-01, 2.4901e-01,
         1.3966e-03],
        [1.4092e-01, 2.8683e-02, 1.5959e-04, 1.0954e-02, 1.8391e-02, 7.9816e-01,
         2.7322e-03],
        [7.6969e-07, 9.9492e-01, 3.9839e-04, 3.6449e-08, 1.4174e-10, 2.3701e-03,
         2.3134e-03],
        [9.6028e-03, 8.9129e-01, 3.2701e-03, 2.7178e-04, 2.5410e-04, 7.8248e-02,
         1.7059e-02],
        [3.5498e-01, 1.4819e-02, 8.2932e-05, 1.5252e-02, 1.0803e-01, 5.0538e-01,
         1.4551e-03]], grad_fn=<SoftmaxBackward0>)

Each row sums to:  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)


### Part-4: Multiply by V

Attention weights tell us WHO to listen to. V contains WHAT they're
actually saying. Multiplying weights × V blends the Value vectors
according to those percentages.

If "love" pays 70% attention to "I" and 30% to "code", its output
becomes 70% of I's Value + 30% of code's Value. The word is now
CONTEXTUALIZED — it carries information from the words it found relevant.
This is the entire point of self-attention.

In [ ]:
output = attention_weights @ V # (3, 3) @ (3, 3) = (3, 3)

print("Final Output:")
print(output)
print("Shape: ", output.shape)

Final Output:
tensor([[ 2.0105, -1.8320, -0.5032],
        [ 1.9831, -1.8293, -0.5625],
        [ 0.9233,  0.1026, -1.2355],
        [ 1.4200, -1.4367, -1.3862],
        [ 2.0101, -1.8324, -0.5059],
        [ 1.9599, -1.8165, -0.5984],
        [ 1.1234, -0.5772, -1.1631]], grad_fn=<MmBackward0>)
Shape:  torch.Size([7, 3])


```python
# =============================================================================================================================
# =============================================================================================================================
# =============================================================================================================================
# =============================================================================================================================
# =============================================================================================================================
```

# Wrapping Everything into a Proper PyTorch Class Using `nn.Module`

Why wrap it in a class?

Right now our code is scattered across multiple cells — embedding here, positional encoding there, attention somewhere else. In real models, everything is organized into modules — self-contained blocks you can reuse, stack, and train.

## Step 1 — What `nn.Module` and `nn.Linear` do

```python
# Quick refresher before we build

# nn.Module = a container for your model. It:
#   - Holds learnable parameters (weights)
#   - Knows how to do a forward pass (input → output)
#   - Lets PyTorch track gradients automatically

# nn.Linear(in_features, out_features) = a layer that does:
#   output = input @ weight.T + bias
#   It's basically matrix multiplication with LEARNABLE weights.
#   This replaces our manual W_Q = torch.randn(...) approach.
```
Remember when we did `Q = final_embeddings @ W_Q`? That's exactly what `nn.Linear` does — but it also manages the weights for you and makes them trainable automatically. We created `W_Q` manually with `torch.randn`. With `nn.Linear`, PyTorch creates it, tracks it, and updates it during training.

## Step 2 — The Embedding + Positional Encoding Module

In [ ]:
import torch
import torch.nn as nn
import math

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=100):
        super().__init__()
        # The lookup table: vocab_size rows, d_model columns
        self.embedding = nn.Embedding(vocab_size, d_model)

        # Positional encoding (not learned — fixed sin/cos)
        pe = torch.zeros(max_len, d_model)
        for pos in range(max_len):
            for i in range(0, d_model, 2):
                denominator = 10000 ** (i / d_model)
                pe[pos, i]     = math.sin(pos / denominator)
                pe[pos, i + 1] = math.cos(pos / denominator)

        # register_buffer = "save this tensor but DON'T train it"
        # because positional encodings are fixed, not learned
        self.register_buffer('pe', pe)

    def forward(self, token_ids):
        # Step 1: Look up token embeddings
        tok_emb = self.embedding(token_ids)

        # Step 2: Add positional encoding
        seq_len = token_ids.shape[0]
        output = tok_emb + self.pe[:seq_len]

        return output

## Step 3 — Self-Attention as a Module

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        # These replace our manual W_Q, W_K, W_V = torch.randn(...)
        # nn.Linear creates the weight matrix AND makes it trainable
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)
        self.d_k = d_k

    def forward(self, x):
        # x shape: (seq_len, d_model) — our final_embeddings

        # Step 1: Project into Q, K, V
        # Same as: Q = final_embeddings @ W_Q
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # Step 2: Dot product — "how relevant is each word to each other?"
        scores = Q @ K.T

        # Step 3: Scale — keep softmax in its useful zone
        scaled_scores = scores / math.sqrt(self.d_k)

        # Step 4: Softmax — convert to probabilities (each row sums to 1)
        attention_weights = torch.softmax(scaled_scores, dim=-1)

        # Step 5: Multiply by V — blend information based on attention
        output = attention_weights @ V

        return output, attention_weights

### What is Bias and Why?

Remember the neuron formula:
```
output = (input × weight) + bias
```

Bias shifts the entire output by a fixed amount. It's like adjusting where your starting point is. Weight controls the slope (how much input affects output), bias controls the intercept (where you start).

### So why does nn.Linear have bias?

`nn.Linear` is a general-purpose layer. It does:
```
output = input @ weight.T + bias
```
In most neural network layers (feed-forward networks, output layers), you want bias because that flexibility helps the model learn better. The neuron needs to be able to say "even when all my inputs are zero, my output should be something non-zero."

### Then why set bias=False for Q, K, V?

For attention specifically, we're computing relative comparisons between words. The dot product `Q @ K.T` measures how similar two vectors are based on their direction. Adding bias would shift ALL queries or ALL keys by the same fixed amount — it doesn't help with comparing words to each other because it affects every word equally.

## Step 4 — Putting It All Together

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, vocab_size, d_model, d_k):
        super().__init__()
        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.self_attention = SelfAttention(d_model, d_k)

    def forward(self, token_ids):
        # Full pipeline: IDs → embeddings → positional → attention
        embeddings = self.token_embedding(token_ids)
        output, attention_weights = self.self_attention(embeddings)
        return output, attention_weights

## Step 5 — Run It

In [ ]:
# Our sentence setup from before
text = "I love code and I love AI"
tokens = text.lower().split()
vocab = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocab)}
token_ids = torch.tensor([word_to_id[t] for t in tokens])

print("Tokens:", tokens)
print("IDs:", token_ids)

# Create the model
model = TransformerBlock(
    vocab_size=len(vocab),   # 5 unique words
    d_model=4,               # embedding dimension
    d_k=3                    # attention dimension
)

# Run it!
output, attention_weights = model(token_ids)

print("\nOutput shape:", output.shape)          # (7, 3)
print("Attention matrix shape:", attention_weights.shape)  # (7, 7)

print("\nContextualized output:")
print(output)

print("\nAttention weights (who pays attention to whom):")
print(attention_weights)

Tokens: ['i', 'love', 'code', 'and', 'i', 'love', 'ai']
IDs: tensor([3, 4, 2, 1, 3, 4, 0])

Output shape: torch.Size([7, 3])
Attention matrix shape: torch.Size([7, 7])

Contextualized output:
tensor([[ 0.1131, -0.3455,  0.6223],
        [ 0.1670, -0.3382,  0.6707],
        [-0.0267, -0.3380,  0.3085],
        [-0.0268, -0.3333,  0.2500],
        [ 0.0009, -0.3373,  0.3978],
        [ 0.0765, -0.3319,  0.5312],
        [-0.0551, -0.3338,  0.2236]], grad_fn=<MmBackward0>)

Attention weights (who pays attention to whom):
tensor([[0.1432, 0.2099, 0.0517, 0.0624, 0.1316, 0.3237, 0.0775],
        [0.1276, 0.2112, 0.0379, 0.0569, 0.1233, 0.3846, 0.0584],
        [0.1450, 0.1351, 0.1691, 0.1509, 0.1361, 0.1140, 0.1498],
        [0.1282, 0.1097, 0.1892, 0.1731, 0.1346, 0.0954, 0.1698],
        [0.1535, 0.1662, 0.1274, 0.1295, 0.1403, 0.1615, 0.1216],
        [0.1457, 0.2012, 0.0738, 0.1015, 0.1367, 0.2618, 0.0794],
        [0.1457, 0.1205, 0.2273, 0.1683, 0.1184, 0.0709, 0.1489]],
       grad_f

## Step 6 — Inspect What the Model Learned

In [ ]:
# Let's peek inside the model

print("=== Learnable parameters ===")
for name, param in model.named_parameters():
    print(f"{name}: shape {param.shape}, trainable: {param.requires_grad}")

print("\n=== Token Embedding Table ===")
print(model.token_embedding.embedding.weight)

print("\n=== Positional Encoding (NOT trainable) ===")
print(model.token_embedding.pe[:7])

=== Learnable parameters ===
token_embedding.embedding.weight: shape torch.Size([5, 4]), trainable: True
self_attention.W_Q.weight: shape torch.Size([3, 4]), trainable: True
self_attention.W_K.weight: shape torch.Size([3, 4]), trainable: True
self_attention.W_V.weight: shape torch.Size([3, 4]), trainable: True

=== Token Embedding Table ===
Parameter containing:
tensor([[ 1.3916, -0.2935,  1.7657,  0.2136],
        [ 0.8646, -0.7861, -0.1675, -1.1580],
        [ 1.4853, -1.2708,  0.2641, -0.0078],
        [ 0.9796,  0.9095,  0.0785, -0.1003],
        [-0.3736, -0.9295, -1.7846,  0.5561]], requires_grad=True)

=== Positional Encoding (NOT trainable) ===
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [-0.7568, -0.6536,  0.0400,  0.9992],
        [-0.9589,  0.2837,  0.0500,  0.9988],
        [-0.2794,  0.9602,  0.0600,  0.9982]])


```python
# =============================================================================================================================
# =============================================================================================================================
# =============================================================================================================================
# =============================================================================================================================
# =============================================================================================================================
```

# Multi-Head Attention

### Why do we need multiple heads?

One attention head produces one set of attention weights — one perspective on how words relate. But language is complex. A single word can have multiple types of relationships simultaneously.

Take the sentence: `"The cat sat on the mat because it was tired"`

The word "it" needs to figure out:
```
Relationship 1: "it" refers to "cat" (not "mat") — coreference
Relationship 2: "it" comes after "because"        — syntactic structure  
Relationship 3: "it" connects to "tired"           — who is tired?
```
One attention head can only learn one pattern of Q-K matching. It might learn to find coreferences, but then it can't simultaneously learn syntactic patterns. That's like having one employee who can only do one job.

Multi-head attention runs several attention heads in parallel, each with its own W_Q, W_K, W_V. Each head is free to learn a different type of relationship. Then we combine their outputs at the end.
```
Single head:  ONE perspective   → limited understanding
Multi-head:   MANY perspectives → rich understanding

Head 1: "Who does this pronoun refer to?"
Head 2: "What's the verb connected to this subject?"
Head 3: "Which adjective modifies this noun?"
Head 4: "What comes next in this phrase?"
```


### The math — how d_k connects to number of heads
`d_k` is smaller than `d_model` to make room for multiple heads.

Here's exactly how:
```
d_model = 8     (each word's full embedding size)
num_heads = 2   (we want 2 perspectives)
d_k = d_model / num_heads = 8 / 2 = 4

Head 1: works in 4 dimensions with its own W_Q, W_K, W_V
Head 2: works in 4 dimensions with its own W_Q, W_K, W_V

Total dimensions used: 4 + 4 = 8 = d_model ✓
```
Each head gets a slice of the full dimensionality. The total compute stays the same as one big head — we're just splitting the work.

In real models:
```
GPT-2:  d_model=768,   12 heads, d_k = 768/12  = 64 per head
GPT-3:  d_model=12288, 96 heads, d_k = 12288/96 = 128 per head
```


## Step 1 — Let's build it

```python
# MULTI-HEAD ATTENTION
#
# One head = one perspective on word relationships
# Multiple heads = multiple perspectives running in parallel
#
# Each head has its own W_Q, W_K, W_V and learns different patterns.
# d_k = d_model / num_heads — each head works in a smaller space.
# After all heads compute attention, their outputs are concatenated
# and projected back to d_model dimensions.
#
# Flow:
# input (7, 8) → split into 2 heads → each head does attention in (7, 4)
# → concatenate outputs → (7, 8) → project back → (7, 8)
```

In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        # d_k is CALCULATED, not chosen by us
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # each head's dimension

        # Each head needs its own Q, K, V projections
        # But instead of making separate nn.Linear for each head,
        # we use ONE big linear layer and then SPLIT the output
        # This is more efficient — one big matrix multiply instead of many small ones
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # (8, 8) not (8, 4)!
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)

        # After concatenating all heads, project back to d_model
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        seq_len = x.shape[0]     # number of tokens (7)

        # Step 1: Project into Q, K, V (full d_model size)
        Q = self.W_Q(x)   # (7, 8)
        K = self.W_K(x)   # (7, 8)
        V = self.W_V(x)   # (7, 8)

        # Step 2: SPLIT into multiple heads
        # Reshape from (7, 8) → (7, 2, 4) → (2, 7, 4)
        # 7 tokens, 2 heads, 4 dimensions per head
        Q = Q.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        K = K.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        V = V.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        # Now Q, K, V are each (2, 7, 4) — 2 heads, 7 tokens, 4 dims per head

        # Step 3: Attention (same formula, now per head)
        scores = Q @ K.transpose(-2, -1)       # (2, 7, 7) — each head has its own 7×7
        scaled_scores = scores / math.sqrt(self.d_k)
        attention_weights = torch.softmax(scaled_scores, dim=-1)  # (2, 7, 7)

        # Step 4: Multiply by V
        head_outputs = attention_weights @ V    # (2, 7, 4)

        # Step 5: CONCATENATE all heads back together
        # (2, 7, 4) → (7, 2, 4) → (7, 8)
        head_outputs = head_outputs.permute(1, 0, 2).contiguous().view(seq_len, self.d_model)

        # Step 6: Final projection
        output = self.W_O(head_outputs)    # (7, 8)

        return output, attention_weights

### Step 2 — Understanding the split

The key insight is the reshape and permute in Step 2:

```python
# Let's trace what happens to Q with actual shapes

# After W_Q projection:
# Q shape: (7, 8) — 7 tokens, 8 dimensions
#
# Token "i":    [a, b, c, d, e, f, g, h]   ← 8 numbers
#
# We SPLIT those 8 numbers into 2 groups of 4:
#   Head 1 gets: [a, b, c, d]   ← first 4 dimensions
#   Head 2 gets: [e, f, g, h]   ← last 4 dimensions

# .view(7, 2, 4) does the split:
# (7, 8) → (7, 2, 4)
# 7 tokens, 2 heads, 4 dims each
#
# Token "i" becomes:
#   Head 1: [a, b, c, d]
#   Head 2: [e, f, g, h]

# .permute(1, 0, 2) rearranges to put heads first:
# (7, 2, 4) → (2, 7, 4)
# 2 heads, 7 tokens, 4 dims each
#
# Now it's organized as:
#   Head 1: all 7 tokens with their first-4-dim queries
#   Head 2: all 7 tokens with their last-4-dim queries
```

### Why Put Heads First?

Imagine we have the Q tensor after splitting:
```
After .view(7, 2, 4):  shape is (7, 2, 4)
                         tokens, heads, dims

Organized as:
  Token 0: [Head1: [a,b,c,d], Head2: [e,f,g,h]]
  Token 1: [Head1: [i,j,k,l], Head2: [m,n,o,p]]
  Token 2: [Head1: [q,r,s,t], Head2: [u,v,w,x]]
  ...  
```

The data is grouped BY TOKEN. But attention needs to compare
tokens WITHIN each head.

After .permute(1, 0, 2) — heads come first:

```
After permute: shape is (2, 7, 4)
                         heads, tokens, dims

Organized as:
  Head 1: [Token0: [a,b,c,d],  Token1: [i,j,k,l],  Token2: [q,r,s,t], ...]
  Head 2: [Token0: [e,f,g,h],  Token1: [m,n,o,p],  Token2: [u,v,w,x], ...]
```
Now the data is grouped BY HEAD. Each head has all 7 tokens together.

Why does this matter? When we do Q @ K.T, PyTorch treats the first dimension as a batch. It says "do separate matrix multiplications for each item in dimension 0":

```
Q @ K.T with shape (2, 7, 4) @ (2, 4, 7):

PyTorch does:
  Batch 0 (Head 1): (7, 4) @ (4, 7) = (7, 7)  ← head 1's attention
  Batch 1 (Head 2): (7, 4) @ (4, 7) = (7, 7)  ← head 2's attention
```

Result: (2, 7, 7) — two separate attention matrices, one per head
If heads were NOT first, PyTorch would try to compare tokens across heads, which is wrong. Head 1's queries should only match with Head 1's keys, not Head 2's keys. Putting heads first makes PyTorch treat each head as an independent attention computation.

### Step 3 — Understanding the concatenation

After attention, each head produced its own output:
```
Head 1 output: (7, 4) — 7 tokens, each with 4-dim contextualized vector
Head 2 output: (7, 4) — 7 tokens, each with 4-dim contextualized vector

Combined: (2, 7, 4)

We reverse the split:
permute(1, 0, 2): (2, 7, 4) → (7, 2, 4)
view(7, 8):       (7, 2, 4) → (7, 8)

Token "i" now has: [head1_a, head1_b, head1_c, head1_d,
                    head2_a, head2_b, head2_c, head2_d]
```

Both perspectives glued together into one 8-dim vector.


### Step 4 — What's W_O and why?
After concatenation, each token has information from multiple heads just stacked side by side. W_O (the output projection) mixes them together:
```
Without W_O: [head1 info | head2 info]  — just glued, no interaction
With W_O:    head1 and head2 info BLENDED into a unified representation
```
It's like having two advisors give you separate reports. W_O is you reading both and writing a unified summary.

### Step 5 — Run It

In [ ]:
# Same sentence setup
text = "I love code and I love AI"
tokens = text.lower().split()
vocab = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocab)}
token_ids = torch.tensor([word_to_id[t] for t in tokens])

# Embedding layer (d_model=8 now, to allow 2 heads of 4 each)
d_model = 8
embedding = nn.Embedding(len(vocab), d_model)


# Create positional encoding
def create_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            denominator = 10000 ** (i / d_model)
            pe[pos, i]     = math.sin(pos / denominator)
            pe[pos, i + 1] = math.cos(pos / denominator)
    return pe

pe = create_positional_encoding(10, d_model)

# Get embeddings + positional encoding
final_embeddings = embedding(token_ids) + pe[:len(tokens)]

# Create multi-head attention (2 heads)
mha = MultiHeadAttention(d_model=8, num_heads=2)

# Run it!
output, attention_weights = mha(final_embeddings)

print("Input shape:", final_embeddings.shape)    # (7, 8)
print("Output shape:", output.shape)              # (7, 8)
print("Attention shape:", attention_weights.shape) # (2, 7, 7) — one 7×7 per head

print("\nHead 1 attention (who pays attention to whom):")
print(attention_weights[0])

print("\nHead 2 attention (DIFFERENT pattern!):")
print(attention_weights[1])

Input shape: torch.Size([7, 8])
Output shape: torch.Size([7, 8])
Attention shape: torch.Size([2, 7, 7])

Head 1 attention (who pays attention to whom):
tensor([[0.1385, 0.2031, 0.0765, 0.0961, 0.1003, 0.2270, 0.1584],
        [0.0992, 0.1019, 0.2938, 0.1509, 0.1769, 0.1024, 0.0750],
        [0.1345, 0.1688, 0.1295, 0.1291, 0.1248, 0.1689, 0.1444],
        [0.1225, 0.2000, 0.1178, 0.1167, 0.1034, 0.1960, 0.1437],
        [0.1116, 0.2680, 0.0511, 0.0778, 0.0554, 0.2613, 0.1747],
        [0.0942, 0.1759, 0.2230, 0.1377, 0.1127, 0.1565, 0.0999],
        [0.1409, 0.1132, 0.1654, 0.1185, 0.2160, 0.1505, 0.0954]],
       grad_fn=<SelectBackward0>)

Head 2 attention (DIFFERENT pattern!):
tensor([[0.2328, 0.0827, 0.0612, 0.0984, 0.1818, 0.1016, 0.2415],
        [0.2012, 0.0686, 0.1596, 0.1482, 0.2170, 0.0768, 0.1286],
        [0.1980, 0.0987, 0.1124, 0.1325, 0.1820, 0.1143, 0.1621],
        [0.2384, 0.0738, 0.0860, 0.1158, 0.2074, 0.0925, 0.1861],
        [0.2202, 0.0953, 0.0688, 0.1057, 0.1746

### Step 6 — Compare the two heads
This is the whole point — look at how the two heads develop different attention patterns:

Even with random weights, the two heads will likely show different attention patterns. In a trained model, these differences would be meaningful — one head might track grammar, another might track meaning.

In [ ]:
print("=== Word 'i' (position 0) attention patterns ===")
print(f"\nHead 1: {attention_weights[0][0].detach()}")
print(f"Head 2: {attention_weights[1][0].detach()}")
print(f"\nHead 1 thinks 'i' should attend most to: {tokens[attention_weights[0][0].argmax()]}")
print(f"Head 2 thinks 'i' should attend most to: {tokens[attention_weights[1][0].argmax()]}")

=== Word 'i' (position 0) attention patterns ===

Head 1: tensor([0.1385, 0.2031, 0.0765, 0.0961, 0.1003, 0.2270, 0.1584])
Head 2: tensor([0.2328, 0.0827, 0.0612, 0.0984, 0.1818, 0.1016, 0.2415])

Head 1 thinks 'i' should attend most to: love
Head 2 thinks 'i' should attend most to: ai


We used one big nn.Linear(d_model, d_model) for W_Q and then split the output into heads, instead of creating separate nn.Linear(d_model, d_k) for each head. Both approaches give the same result. Why is the "one big matrix, then split" approach preferred?

One big multiplication is much faster on GPUs than many small ones because GPUs are designed to handle large parallel operations. Doing 12 separate small multiplications has overhead — launching each operation, moving data around — while one big one flows through the hardware smoothly.